# 02 - Cost & Utility Computation (Normalized Schema)

This notebook works with the **normalized 4-table schema**:
- Reads from `vlm_samples` + `vlm_responses` + `vlm_evaluations`
- Computes sample scores, priors, and utility per model
- Updates `vlm_responses` with computed scores

In [ ]:
# === Path Setup ===
import sys
from pathlib import Path

# Notebook paths - all ares notebooks are in artemis_final/notebooks/ares/
NOTEBOOK_DIR = Path.cwd()
ARTEMIS_DIR = NOTEBOOK_DIR.parent.parent  # artemis_final/
ROOT_DIR = ARTEMIS_DIR.parent             # Which_VLM_Router/

# Add to sys.path for imports
for p in [str(ARTEMIS_DIR), str(ROOT_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"📁 ARTEMIS_DIR: {ARTEMIS_DIR}")

import numpy as np
import pandas as pd
from scipy.special import softmax
from sqlalchemy import text
from tqdm.auto import tqdm

from ares.db.connection import get_engine, test_connection
from ares.configs.db_config import TABLES, MODEL_NAMES
from ares.utils.common_utils import return_model_pricing

print('Imports successful!')

In [ ]:
# Configuration
W_CORRECT = 0.45
W_F1 = 0.05
W_GLIDER = 0.5

W_SAMPLE = 0.4
W_TASK = 0.6
W_GLOBAL = 0.05

ALPHA = 50.0
LAMBDA_COST = 10000

engine = get_engine()
test_connection()

In [ ]:
# Load joined data from normalized tables
samples_table = TABLES['samples']
responses_table = TABLES['responses']
evaluations_table = TABLES['evaluations']

query = f'''
SELECT 
    s.sample_id, s.source_config, s.router_task, s.data_split,
    r.model_name, r.is_correct, r.score_f1, r.input_tokens, r.output_tokens,
    r.estimated_cost_usd, r.ok,
    e.glider_score, e.semantic_f1_f1
FROM {samples_table} s
JOIN {responses_table} r ON s.sample_id = r.sample_id
LEFT JOIN {evaluations_table} e ON r.sample_id = e.sample_id AND r.model_name = e.model_name
ORDER BY s.sample_id, r.model_name
'''
df = pd.read_sql(query, engine)
print(f'Loaded {len(df)} rows ({df["sample_id"].nunique()} samples x {df["model_name"].nunique()} models)')
df.head()

In [ ]:
# Compute sample scores
def compute_sample_score(row):
    is_correct = float(row['is_correct'] or 0)
    score_f1 = float(row['score_f1'] or 0)
    glider = float(row['glider_score'] or 1)
    glider_norm = (glider - 1) / 4  # [1,5] -> [0,1]
    return W_CORRECT * is_correct + W_F1 * score_f1 + W_GLIDER * glider_norm

df['sample_score'] = df.apply(compute_sample_score, axis=1)
print('Sample scores computed')
df.groupby('model_name')['sample_score'].mean()

In [ ]:
# Compute global and task priors from TRAINING data only
df_train = df[df['data_split'] == 'train']

global_priors = df_train.groupby('model_name')['sample_score'].mean().to_dict()
print('Global priors:')
for m, p in global_priors.items():
    print(f'  {m}: {p:.4f}')

# Task priors with smoothing
task_priors = {}
for task in df_train['router_task'].unique():
    task_df = df_train[df_train['router_task'] == task]
    n = len(task_df) / len(MODEL_NAMES)  # samples per model
    for model in MODEL_NAMES:
        model_df = task_df[task_df['model_name'] == model]
        task_mean = model_df['sample_score'].mean() if len(model_df) > 0 else global_priors.get(model, 0.5)
        g = global_priors.get(model, 0.5)
        smoothed = (n * task_mean + ALPHA * g) / (n + ALPHA)
        task_priors[(task, model)] = smoothed

print(f'Computed task priors for {len(df_train["router_task"].unique())} tasks')

In [ ]:
# Compute hierarchical performance
def get_perf_hier(row):
    sample = row['sample_score']
    task = row['router_task']
    model = row['model_name']
    g = global_priors.get(model, 0.5)
    t = task_priors.get((task, model), g)
    return W_SAMPLE * sample + W_TASK * t + W_GLOBAL * g

df['perf_hier'] = df.apply(get_perf_hier, axis=1)
print('Hierarchical performance:')
df.groupby('model_name')['perf_hier'].mean()

In [ ]:
# Normalize costs
costs = df['estimated_cost_usd'].fillna(0)
c_min, c_max = costs.quantile(0.02), costs.quantile(0.98)
df['cost_norm'] = ((costs - c_min) / (c_max - c_min + 1e-9)).clip(0, 1)

# Compute utility
df['utility'] = df['perf_hier'] - LAMBDA_COST * df['cost_norm']
print('Utility by model:')
df.groupby('model_name')['utility'].mean().sort_values(ascending=False)

In [ ]:
# Select best model per sample
best_idx = df[df['utility'].notna()].groupby('sample_id')['utility'].idxmax()
if len(best_idx) == 0:
    print('No utility values available to pick best models.')
    best = pd.DataFrame(columns=['sample_id', 'best_model'])
else:
    best = df.loc[best_idx][['sample_id', 'model_name']]
    best.columns = ['sample_id', 'best_model']
    print('Best model distribution:')
    print(best['best_model'].value_counts())


In [ ]:
# Compute soft labels (probability per model)
pivot = df.pivot(index='sample_id', columns='model_name', values='utility')
soft = pd.DataFrame(
    softmax(pivot.values, axis=1),
    columns=[f'soft_p_{m}' for m in pivot.columns],
    index=pivot.index
)
print('Soft labels computed')
soft.head()

In [ ]:
# Update vlm_responses with sample_score, perf_hier, utility
update_sql = text('''
    UPDATE vlm_responses SET
        sample_score = :sample_score,
        perf_hier = :perf_hier,
        cost_norm = :cost_norm,
        utility = :utility,
        updated_at = NOW()
    WHERE sample_id = :sample_id AND model_name = :model_name
''')

with engine.begin() as conn:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='Updating'):
        conn.execute(update_sql, {
            'sample_score': row['sample_score'],
            'perf_hier': row['perf_hier'],
            'cost_norm': row['cost_norm'],
            'utility': row['utility'],
            'sample_id': row['sample_id'],
            'model_name': row['model_name'],
        })

print('Done!')